# Protocol 7 — Disorders of Consciousness Biomarker

Simulates predictions Pred 7.A–Pred 7.E from the APGI clinical protocol
(*Disorders of Consciousness Biomarker: Joint Roles of Ignition Capacity and Interoceptive Precision*):

* **Pred 7.A** — HEP + PCI joint model explains more variance in 3-month **CRS-R** total score outcome than either biomarker alone (ΔR² ≥ 0.05; LRT p < 0.05; AUC > 0.80 aspirational secondary metric)
* **Pred 7.B** — HEP amplitude discriminates MCS from VS/UWS (d > 0.5); four-group gradient VS/UWS < MCS < EMCS < controls confirmed for both HEP and PCI; DMN↔PCI r > 0.50 AND DMN↔HEP r < 0.20 (double dissociation)
* **Pred 7.C** — Interoceptive perturbation via CCRC (5 % CO₂/95 % O₂, 90 s) increases PCI ≥ 10 % in MCS but not VS/UWS; PCI change correlates with HEP change (r > 0.4); CCRC-evoked ΔPCI exceeds white-noise control
* **Pred 7.D** — HEP amplitude correlates with **CRS-R** total score at 3-month (primary) and 6-month (primary) follow-up (within-participant Spearman r > 0.4; JMbayes2 joint modelling; LOCF explicitly excluded)
* **Pred 7.E** *(exploratory)* — Somatic bias modulates reportable embodiment weighting toward 'bodily/visceral' dimensions (d > 0.45)

> **EP-7 | Paper 3 | Clinical | Depends on: EP-0 | N target = 110 (VS/UWS = 30, MCS = 30, EMCS = 20, controls = 30)**
>
> Primary outcome measure: **CRS-R total score** (Coma Recovery Scale-Revised). NOT GCS-S.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from apgi.core import compute_pi_i_eff, BETA_SM_DEFAULT

## 1 — Protocol 7 parameters

In [ ]:
# Estimated Πⁱ by DoC group — from protocol_7_doc_biomarker.json → apgi_parameters
# N per group from participants.n_per_group in protocol_7_doc_biomarker.json
PI_I_BY_GROUP = {
    "VS/UWS": 0.3,    # pi_i_estimated_VS_UWS
    "MCS":    0.8,    # pi_i_estimated_MCS
    "EMCS":   1.1,    # pi_i_estimated_EMCS
    "Controls": 1.2,  # pi_i_estimated_controls
}
N_PER_GROUP = {
    "VS/UWS":   30,   # n_per_group.vegetative_state_UWS
    "MCS":      30,   # n_per_group.minimally_conscious_state
    "EMCS":     20,   # n_per_group.emerging_minimally_conscious_state_EMCS
    "Controls": 30,   # n_per_group.healthy_controls
}
# N_TOTAL = 110 (n_target)
KAPPA = 100.0
# Primary outcome: CRS-R total score (NOT GCS-S)

rng = np.random.default_rng(7)
hep_by_group, pci_by_group = {}, {}
for group, pi_i in PI_I_BY_GROUP.items():
    n = N_PER_GROUP[group]
    M_hat = rng.uniform(0.0, 0.5, n)
    pi_eff = np.array([compute_pi_i_eff(pi_i, BETA_SM_DEFAULT, float(M_hat[k])) for k in range(n)])
    hep_by_group[group] = pi_eff + rng.normal(0, 0.07, n)
    pci_base = {"VS/UWS": 0.18, "MCS": 0.35, "EMCS": 0.44, "Controls": 0.52}[group]
    pci_by_group[group] = rng.normal(pci_base, 0.06, n)

for g in PI_I_BY_GROUP:
    print(f"{g:<12} HEP={hep_by_group[g].mean():.3f}  PCI={pci_by_group[g].mean():.3f}")

## 2 — Prediction 7.B: HEP and PCI four-group gradient (VS/UWS < MCS < EMCS < Controls)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
groups = list(PI_I_BY_GROUP.keys())  # ["VS/UWS", "MCS", "EMCS", "Controls"]
colors = ["#d6604d", "#FFCC00", "#66CC99", "#2166ac"]
x = range(len(groups))

ax = axes[0]
ax.bar(
    x,
    [hep_by_group[g].mean() for g in groups],
    yerr=[hep_by_group[g].std() / np.sqrt(N_PER_GROUP[g]) for g in groups],
    color=colors,
    alpha=0.85,
    edgecolor="white",
    capsize=5,
    width=0.5,
)
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylabel("HEP amplitude (a.u.)")
ax.set_title("Pred 7.B — HEP four-group gradient\n(VS/UWS < MCS < EMCS < Controls)")
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
ax.bar(
    x,
    [pci_by_group[g].mean() for g in groups],
    yerr=[pci_by_group[g].std() / np.sqrt(N_PER_GROUP[g]) for g in groups],
    color=colors,
    alpha=0.85,
    edgecolor="white",
    capsize=5,
    width=0.5,
)
ax.axhline(0.31, ls="--", lw=1, color="k", alpha=0.5, label="PCI threshold (0.31)")
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylabel("PCI")
ax.set_title("PCI four-group gradient\n(VS/UWS < MCS < EMCS < Controls)")
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Protocol 7 — Biomarkers across DoC groups (N=110: VS/UWS=30, MCS=30, EMCS=20, Controls=30)", fontsize=11)
plt.tight_layout()
plt.show()

## 3 — Prediction 7.A: Joint model ΔR² vs. univariate models (outcome = CRS-R total score)

In [ ]:
all_hep, all_pci, all_crs_r = [], [], []
# CRS-R total score ranges 0–23; typical values: VS/UWS ~4, MCS ~9, EMCS ~18, controls ~23
base_crs_r = {"VS/UWS": 4.0, "MCS": 9.0, "EMCS": 18.0, "Controls": 23.0}
for group in groups:
    n = N_PER_GROUP[group]
    crs_r = (
        base_crs_r[group]
        + 2.0 * hep_by_group[group]
        + 3.0 * pci_by_group[group]
        + rng.normal(0, 1.2, n)
    )
    all_hep.extend(hep_by_group[group])
    all_pci.extend(pci_by_group[group])
    all_crs_r.extend(crs_r)
all_hep   = np.array(all_hep)
all_pci   = np.array(all_pci)
all_crs_r = np.array(all_crs_r)


def r2(y, *Xs):
    X = np.column_stack([np.ones(len(y))] + list(Xs))
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    ss_res = np.sum((y - X @ b) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot


r2_hep   = r2(all_crs_r, all_hep)
r2_pci   = r2(all_crs_r, all_pci)
r2_joint = r2(all_crs_r, all_hep, all_pci)
delta_r2 = r2_joint - max(r2_hep, r2_pci)

print(f"R² HEP only:    {r2_hep:.3f}")
print(f"R² PCI only:    {r2_pci:.3f}")
print(f"R² Joint model: {r2_joint:.3f}")
print(f"ΔR² (joint vs. best univariate): {delta_r2:.3f}  (Pred 7.A threshold: ΔR² ≥ 0.05 → {'PASS' if delta_r2 >= 0.05 else 'FAIL'})")

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ["HEP only", "PCI only", "Joint"],
    [r2_hep, r2_pci, r2_joint],
    color=["#2166ac", "#9966FF", "#00CC99"],
    alpha=0.85,
    edgecolor="white",
    width=0.4,
)
for bar, v in zip(bars, [r2_hep, r2_pci, r2_joint]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        v + 0.01,
        f"{v:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set_ylabel("R² for CRS-R total score (3-month)")
ax.set_title("Pred 7.A — Joint model superiority\n(ΔR² ≥ 0.05 required for confirmation)")
ax.set_ylim(0, 1)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()